# Toric code NES test

This notebook tests the sampled NES implementation on the 2D toric code. The toric code lives on **edges**, so `shape=(Lx,Ly)` means `N_edges=2*Lx*Ly` spin variables. On a torus, the ground state is four-fold degenerate.


In [ ]:
import sys
from pathlib import Path

# Works whether the notebook is launched from project root or notebooks/.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from nes_lattice.train import TrainConfig, train, save_history
from nes_lattice.hamiltonians import make_hamiltonian_spec
from nes_lattice.references import get_reference_energies
from nes_lattice.plots import plot_history, plot_diagnostics, print_final

PROJECT_ROOT


## Check toric-code terms and exact ground degeneracy

In [ ]:
hspec = make_hamiltonian_spec(name='toric_code', shape=(2, 2), Je=1.0, Jm=1.0, pbc=True)
terms = hspec.bonds_np
stars, plaquettes = terms[0], terms[1]
print('shape:', hspec.shape)
print('edge spins N:', hspec.N)
print('number of stars:', len(stars))
print('number of plaquettes:', len(plaquettes))
print('first star:', stars[0])
print('first plaquette:', plaquettes[0])

ref, msg = get_reference_energies(hspec, k=4, prefer='auto')
print('reference source:', msg)
print('first 4 reference energies:', ref)


## Example A: 2x2 toric code, k=4, FFN

This is the first correctness test. Since `N_edges=8`, exact span evaluation and dense ED are cheap. The target is four energies close to `-8` for `Je=Jm=1`.


In [ ]:
cfg = TrainConfig(
    shape=(2, 2),
    hamiltonian='toric_code',
    k=4,
    Je=1.0,
    Jm=1.0,
    model='ffn',
    hidden=(64, 64),
    steps=1500,
    lr=2e-3,
    n_chains=96,
    n_samples=8,
    print_every=100,
    eval_exact_if_sites_leq=12,
    reference='auto',
    seed=0,
)

params, history = train(cfg)

save_path = PROJECT_ROOT / 'results' / 'sampled_nes_toric_2x2_k4_ffn.json'
save_history(history, save_path, cfg)

print('saved to:', save_path)


In [ ]:
print_final(save_path)
fig, ax = plot_history(save_path)
fig


In [ ]:
(fig_cond, ax_cond), (fig_diag, ax_diag) = plot_diagnostics(save_path)
fig_cond


In [ ]:
fig_diag


## Example B: 3x3 toric code, k=4, CNN

This is closer to the intended 2D usage. `shape=(3,3)` has `N_edges=18`, so exact evaluation over `2^18` configs is skipped by setting `eval_exact_if_sites_leq=12`. The analytic toric-code reference still gives the four degenerate ground energies `E=-18` for `Je=Jm=1`.


In [ ]:
cfg = TrainConfig(
    shape=(3, 3),
    hamiltonian='toric_code',
    k=4,
    Je=1.0,
    Jm=1.0,
    model='cnn',
    channels=(16, 16),
    kernel_size=3,
    steps=2000,
    lr=1e-3,
    n_chains=128,
    n_samples=8,
    print_every=100,
    eval_exact_if_sites_leq=12,
    eval_chains=128,
    eval_samples=32,
    reference='auto',
    seed=1,
)

params, history = train(cfg)

save_path = PROJECT_ROOT / 'results' / 'sampled_nes_toric_3x3_k4_cnn.json'
save_history(history, save_path, cfg)

print('saved to:', save_path)


In [ ]:
print_final(save_path)
fig, ax = plot_history(save_path)
fig


In [ ]:
(fig_cond, ax_cond), (fig_diag, ax_diag) = plot_diagnostics(save_path)
fig_cond


In [ ]:
fig_diag
